# Course notebook — Training, Serving, and AI Operations

Understand model adaptation, efficient inference, deployment, observability, release, and lifecycle operations.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Model adaptation and post-training


### Lecture 01 · Choosing Adaptation

Diagnose whether a requirement needs prompting, retrieval, tools, fine-tuning, continued pretraining, or a different model.

**Predict before running.** Choose the smallest intervention at the correct system layer. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Create a decision table for ten adaptation scenarios.


In [ ]:
"""Lab 11.1: Choosing Adaptation"""

CHAPTER = "11.1"
print("chapter hook:", CHAPTER)
scenarios = [
    ("new policy fact", "RAG"),
    ("consistent tone", "prompt/SFT"),
    ("live database count", "tool"),
]
for need, fix in scenarios:
    print({"need": need, "intervention": fix})
print("---")
print("change one input above, predict output, re-run")


### Lecture 02 · Post-Training Methods

Understand supervised fine-tuning, LoRA, QLoRA, preference data, RLHF, DPO, distillation, and model merging.

**Predict before running.** Adaptation trades generality and operational simplicity for targeted behavior. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Fine-tune a small model and evaluate held-out behavior.


In [ ]:
"""Lab 11.2: Post-Training Methods"""

CHAPTER = "11.2"
print("chapter hook:", CHAPTER)
methods = {"prompt": 0.82, "SFT": 0.91, "DPO": 0.93}
cost = {"prompt": 1, "SFT": 4, "DPO": 6}
target = 0.90
choice = min((m for m, s in methods.items() if s >= target), key=lambda m: cost[m])
print({"method": choice, "score": methods[choice]})
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Specify a behavior where prompt changes are a reasonable baseline before considering an adapter.


## Datasets and inference infrastructure


### Lecture 03 · Dataset Engineering

Curate, label, deduplicate, filter, balance, version, document, and protect training and evaluation data.

**Predict before running.** Data design is model behavior design. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Create a data card and contamination check for a small dataset.


In [ ]:
"""Lab 11.3: Dataset Engineering"""

CHAPTER = "11.3"
print("chapter hook:", CHAPTER)
train = {"case-101", "case-102", "case-103"}
eval = {"case-103", "case-200"}
leak = train & eval
print({"leaked_ids": sorted(leak)})
print("---")
print("change one input above, predict output, re-run")


### Lecture 04 · Inference Infrastructure

Connect accelerators, memory, quantization, model formats, servers, batching, streaming, caches, and speculative decoding.

**Predict before running.** Inference performance is a queueing and memory problem as much as a model problem. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Load-test a local model at several concurrency levels.


In [ ]:
"""Lab 11.4: Inference Infrastructure"""

CHAPTER = "11.4"
print("chapter hook:", CHAPTER)
batch_sizes = [1, 4, 8]
for b in batch_sizes:
    throughput = b / (1 + 0.1 * (b - 1))
    print(f"batch={b} relative_throughput={throughput:.2f}")
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Describe a held-out slice and a serving constraint. Explain which one affects training versus deployment.


## Deployment and operations


### Lecture 05 · Deployment and Routing

Design containers, serverless endpoints, Kubernetes, autoscaling, routing, fallbacks, regional placement, and disaster recovery.

**Predict before running.** Deployment choices allocate control, cost, latency, and operational burden. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Write a deployment ADR comparing hosted and self-hosted inference.


In [ ]:
"""Lab 11.5: Deployment and Routing"""

CHAPTER = "11.5"
print("chapter hook:", CHAPTER)
options = {"hosted": {"control": 2, "cost": 3}, "self": {"control": 5, "cost": 4}}
need = "data residency strict"
choice = "self" if "residency" in need else "hosted"
print({"choice": choice, **options[choice]})
print("---")
print("change one input above, predict output, re-run")


### Lecture 06 · LLMOps

Version prompts, models, data, and evals; trace requests; monitor quality and cost; canary, roll back, and respond to incidents.

**Predict before running.** Every production change needs evidence, observability, and a reversible release path. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Instrument a request and inject provider, retrieval, and validation failures.


In [ ]:
"""Lab 11.6: LLMOps"""

CHAPTER = "11.6"
print("chapter hook:", CHAPTER)
canary = {"success_rate": 0.79, "baseline": 0.85, "threshold": -0.03}
delta = canary["success_rate"] - canary["baseline"]
action = "rollback" if delta < canary["threshold"] else "promote"
print({"delta": round(delta, 3), "action": action})
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Compare routing choices and write a rollback trigger in terms of latency and quality.


## Course workshop

Compare the router's decisions for a simple task and a high-risk task. Draft a separate held-out evaluation for any future fine-tune.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Policy-based model routing with risk and fallback controls."""
MODELS = {
    "small": {"quality": 0.75, "cost": 1},
    "large": {"quality": 0.93, "cost": 8},
}


def route(task):
    if task["risk"] == "high" or task["complexity"] > 0.7: return "large"
    return "small"


for task in ({"risk": "low", "complexity": .2}, {"risk": "high", "complexity": .3}):
    model = route(task)
    print(task, "->", model, MODELS[model])


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
assert route({"complexity": 0.2, "risk": "low"}) != route({"complexity": 0.2, "risk": "high"})
print('Workshop checks passed')


## Applied exercise for Lecture 02 — Train a rank-one adapter and evaluate held-out inputs

A frozen two-output model returns the input vector. Fit two small vectors so their outer product adds a rank-one correction. Compare base and adapted error on held-out inputs, then reduce the training steps and rerun. This exercise uses a tiny linear model to make the update visible. A transformer has many more weights and training choices.


In [ ]:
training = [(1.0, 0.0), (0.0, 1.0), (-1.0, -1.0), (2.0, 1.0)]
holdout = [(2.0, -1.0), (-2.0, 3.0)]


def target(point):
    x, y = point
    signal = 2 * x - y
    return [x + signal, y - 0.5 * signal]


def base(point):
    return list(point)


a = [0.1, -0.1]
b = [0.0, 0.0]


def adapted(point):
    hidden = sum(weight * value for weight, value in zip(a, point))
    return [value + weight * hidden for value, weight in zip(base(point), b)]


for _ in range(200):
    for point in training:
        residual = [prediction - wanted for prediction, wanted in zip(adapted(point), target(point))]
        hidden = sum(weight * value for weight, value in zip(a, point))
        gradient_hidden = sum(error * weight for error, weight in zip(residual, b))
        a = [weight - 0.04 * gradient_hidden * value for weight, value in zip(a, point)]
        b = [weight - 0.04 * error * hidden for weight, error in zip(b, residual)]


def squared_error(predict):
    return sum(
        (observed - expected) ** 2
        for point in holdout
        for observed, expected in zip(predict(point), target(point))
    )


base_error = squared_error(base)
adapted_error = squared_error(adapted)
print(f"Held-out squared error: base={base_error:.3f}, adapted={adapted_error:.3f}")
assert adapted_error < 0.1 * base_error


## Course project

Adapt and serve a small model, benchmark it, instrument it, and release it with rollback evidence.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
